# Cluster GRASP pocket scores and report pocket residues

This notebook reads per-atom GRASP scores from the PDB **B-factor column**, clusters atoms with `site_metrics.cluster_atoms_average`, and reports both a center and residue IDs for every retained pocket. It can run locally or in Google Colab. Use the controls at the bottom and click **Run clustering**; results are also written as CSV/text files.

## 0. Install dependencies (run this first in Colab)

This installs every direct notebook dependency and the scientific packages imported by `site_metrics.py`. Colab may already provide some of them; `%pip` will reuse compatible installed versions.

In [ ]:
%pip install -q numpy pandas biopython scikit-learn scipy MDAnalysis networkx tqdm joblib ipywidgets

In [ ]:
# A Colab runtime does not automatically contain files beside the notebook.
# If needed, this prompts once to upload site_metrics.py and the scored PDB file.
from pathlib import Path
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

required_example_files = ('site_metrics.py', '6TY3_probs.pdb')
missing_files = [name for name in required_example_files if not Path(name).is_file()]
if IN_COLAB and missing_files:
    print('Upload site_metrics.py and your *_probs.pdb file (the example expects 6TY3_probs.pdb).')
    files.upload()
elif missing_files:
    print('Local note: missing ' + ', '.join(missing_files))
else:
    print('Required example files are available.')

## 1. Imports

The notebook should be run from this folder so that `site_metrics.py` and the scored PDB are available.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
import site_metrics as grps

## 2. Load coordinates, scores, and residue metadata

Atom order is kept identical across the coordinate, score, and metadata arrays. Residue IDs include chain, residue name, sequence number, and insertion code (when present), for example `X:ARG35` or `A:SER42B`.

In [ ]:
def load_scored_pdb(pdb_file):
    pdb_file = Path(pdb_file)
    if not pdb_file.is_file():
        raise FileNotFoundError(f'Cannot find scored PDB: {pdb_file}')

    structure = PDBParser(QUIET=True).get_structure(pdb_file.stem, str(pdb_file))
    model = next(structure.get_models())
    coords, scores, atom_records = [], [], []

    for chain in model:
        chain_id = chain.id.strip() or '_'
        for residue in chain:
            hetero_flag, resseq, insertion_code = residue.id
            insertion_code = insertion_code.strip()
            residue_id = f'{chain_id}:{residue.resname}{resseq}{insertion_code}'
            for atom in residue:
                score = float(atom.get_bfactor())
                coords.append(atom.get_coord())
                scores.append(score)
                atom_records.append({
                    'chain': chain_id, 'resname': residue.resname,
                    'resid': int(resseq), 'insertion_code': insertion_code,
                    'residue_id': residue_id, 'atom_name': atom.name,
                    'hetero_flag': hetero_flag.strip(), 'grasp_score': score,
                })

    all_coords = np.asarray(coords, dtype=float)
    grasp_scores = np.asarray(scores, dtype=float)
    # site_metrics expects the GRASP score in column 1.
    predicted_probs = np.column_stack((1.0 - grasp_scores, grasp_scores))
    return all_coords, predicted_probs, pd.DataFrame(atom_records)

## 3. Cluster, summarize, and export

Pocket ranks below are recomputed from the requested score aggregation so that rank 1 is always the highest-scoring retained pocket. A residue belongs to a pocket when at least one of its atoms passes the GRASP threshold and is assigned to that pocket.

In [ ]:
def run_pocket_clustering(pdb_file, score_threshold=0.30, distance_threshold=20.0,
                          min_atom_count=10, score_type='mean', output_prefix=None):
    all_coords, predicted_probs, atom_table = load_scored_pdb(pdb_file)
    bind_coords, cluster_ids, _ = grps.cluster_atoms_average(
        all_coords, predicted_probs, threshold=float(score_threshold),
        score_type=score_type, distance_threshold=float(distance_threshold),
    )

    if bind_coords is None:
        empty = pd.DataFrame()
        print(f'No atoms have a GRASP score > {score_threshold:.2f}. Try a lower threshold.')
        return empty, empty

    selected = atom_table.loc[predicted_probs[:, 1] > score_threshold].copy().reset_index(drop=True)
    selected['cluster_id'] = np.asarray(cluster_ids, dtype=int)
    selected[['x', 'y', 'z']] = bind_coords

    groups = []
    for cluster_id, atoms in selected.groupby('cluster_id'):
        if len(atoms) < int(min_atom_count):
            continue
        score = atoms['grasp_score'].agg(score_type) if score_type in ('mean', 'sum') else (atoms['grasp_score'] ** 2).sum()
        groups.append((cluster_id, atoms, float(score)))

    groups.sort(key=lambda item: item[2], reverse=True)
    pocket_rows, residue_rows = [], []
    for rank, (cluster_id, atoms, pocket_score) in enumerate(groups, start=1):
        center = atoms[['x', 'y', 'z']].mean().to_numpy()
        residue_stats = (atoms.groupby(['chain', 'resname', 'resid', 'insertion_code', 'residue_id'], dropna=False)
                         .agg(selected_atom_count=('atom_name', 'size'),
                              max_grasp_score=('grasp_score', 'max'),
                              mean_grasp_score=('grasp_score', 'mean'))
                         .reset_index().sort_values(['chain', 'resid', 'insertion_code']))
        residue_ids = residue_stats['residue_id'].tolist()
        pocket_rows.append({
            'pocket_rank': rank, 'pocket_score': pocket_score,
            'center_x': center[0], 'center_y': center[1], 'center_z': center[2],
            'selected_atom_count': len(atoms), 'residue_count': len(residue_ids),
            'residue_ids': ';'.join(residue_ids),
        })
        residue_stats.insert(0, 'pocket_rank', rank)
        residue_rows.extend(residue_stats.to_dict('records'))

    pockets = pd.DataFrame(pocket_rows)
    residues = pd.DataFrame(residue_rows)
    if output_prefix is None:
        output_prefix = str(Path(pdb_file).with_suffix('')) + '_grasp_pockets'
    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)
    pockets.to_csv(f'{output_prefix}.csv', index=False, float_format='%.4f')
    residues.to_csv(f'{output_prefix}_residues.csv', index=False, float_format='%.4f')
    if len(pockets):
        np.savetxt(f'{output_prefix}_centers.txt', pockets[['center_x', 'center_y', 'center_z']], fmt='%.4f')
    else:
        Path(f'{output_prefix}_centers.txt').write_text('')

    print(f'{len(selected):,} atoms passed the threshold; retained {len(pockets)} pockets.')
    print(f'Wrote {output_prefix}.csv, {output_prefix}_residues.csv, and {output_prefix}_centers.txt')
    return pockets, residues

## 4. Friendly controls

- **GRASP score cutoff**: atoms must score strictly above this value. Start near 0.30; raise it for more confident/smaller pockets.
- **Pocket merge distance**: average-linkage distance in Å. Raise it to merge nearby groups; lower it to split them.
- **Minimum selected atoms**: removes tiny clusters after clustering.

The output prefix may include a folder (for example `results/6TY3`).

In [ ]:
pdb_file_control = widgets.Text(value='6TY3_probs.pdb', description='Scored PDB:', style={'description_width': '140px'}, layout=widgets.Layout(width='520px'))
score_control = widgets.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.01, description='GRASP score cutoff:', readout_format='.2f', continuous_update=False, style={'description_width': '140px'}, layout=widgets.Layout(width='520px'))
distance_control = widgets.FloatSlider(value=20.0, min=2.0, max=40.0, step=0.5, description='Merge distance (Å):', continuous_update=False, style={'description_width': '140px'}, layout=widgets.Layout(width='520px'))
min_atoms_control = widgets.IntSlider(value=10, min=1, max=100, step=1, description='Minimum atoms:', continuous_update=False, style={'description_width': '140px'}, layout=widgets.Layout(width='520px'))
score_type_control = widgets.Dropdown(options=[('Mean atom score', 'mean'), ('Sum of scores', 'sum'), ('Sum of squared scores', 'square')], value='mean', description='Pocket ranking:', style={'description_width': '140px'}, layout=widgets.Layout(width='520px'))
prefix_control = widgets.Text(value='6TY3_grasp_pockets', description='Output prefix:', style={'description_width': '140px'}, layout=widgets.Layout(width='520px'))
run_button = widgets.Button(description='Run clustering', button_style='primary', icon='play')
output = widgets.Output()

def on_run_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            pockets, residues = run_pocket_clustering(
                pdb_file_control.value, score_control.value, distance_control.value,
                min_atoms_control.value, score_type_control.value, prefix_control.value,
            )
            if len(pockets):
                display(Markdown('### Pocket summary'))
                display(pockets.style.format({'pocket_score': '{:.4f}', 'center_x': '{:.3f}', 'center_y': '{:.3f}', 'center_z': '{:.3f}'}))
                display(Markdown('### Residues by pocket'))
                for pocket_rank in pockets['pocket_rank']:
                    pocket_residues = residues.loc[residues['pocket_rank'] == pocket_rank, 'residue_id'].tolist()
                    display(Markdown(f'**Pocket {pocket_rank}:** ' + ', '.join(pocket_residues)))
        except Exception as exc:
            print(f'{type(exc).__name__}: {exc}')

run_button.on_click(on_run_clicked)
controls = widgets.VBox([pdb_file_control, score_control, distance_control, min_atoms_control, score_type_control, prefix_control, run_button])
display(controls, output)